# ── Pipeline 1 ──

In [43]:
from pathlib import Path
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from ollama import chat, ChatResponse

DATA_PATH = Path("disaster_info_with_embeddings.csv")

# multi-qa-mpnet-base-dot-v1 is trained specifically for semantic search / QA
# retrieval (768-dim, dot-product similarity) — higher recall@k than all-MiniLM-L6-v2.
# Must match the model used to build disaster_info_with_embeddings.csv.
EMBEDDING_MODEL = "sentence-transformers/multi-qa-mpnet-base-dot-v1"

In [44]:
def make_embedder():
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

hf = make_embedder()

In [45]:
df1 = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")


In [46]:
import ast
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_event(query, embeddings, events):
    query_embedding = hf.embed_documents([query])[0]
    parsed_embeddings = [ast.literal_eval(e) for e in embeddings]
    similarities = cosine_similarity([query_embedding], parsed_embeddings)
    most_similar_idx = similarities.argmax()
    return events[most_similar_idx]

# Change this query to test different lookups:
# Examples: "Tell me about the disaster in Japan."
#           "Earthquake in Haiti in 2010"
#           "Hurricane in the United States in 2005"
query = "Tell me about the disaster in Japan."
relevant_event = retrieve_relevant_event(query, df1["embeddings"].tolist(), df1["combined"].tolist())
print(relevant_event)

A disaster occurred in Japan (Eastern Asia), from 2019-7-22 to 2019-7-30. Classification: Natural > Meteorological > Extreme temperature. No financial impact data recorded.


In [47]:
response: ChatResponse = chat(model="llama3.1:8b", messages = [
    {"role": "system", "content": "You are a helpful assistant who knows everything about disasters. Please summarize the data."},
    {"role": "user", "content": f"Can you provide the summary about this disaster: {relevant_event}"}
])

print(response.message.content)

#Can you provide more details about this disaster: {relevant_event}. I want to know more about it in a timeline format, including government initiatives, rescue process, and damage costs.
#Your task is to generate a detailed timeline of events surrounding the disaster, including the following data points: the start and end of the disaster, government initiatives on specific dates, the rescue process, the cost of damage and renovation, aid and insurance costs.


Based on my research, I found that the disaster you're referring to is likely the 2019 heat wave in Japan.

Here's a summary of the event:

**Disaster Summary**

* **Date:** July 22-30, 2019
* **Location:** Japan (Eastern Asia)
* **Classification:** Natural > Meteorological > Extreme temperature
* **Impact:**
	+ High temperatures reached up to 40°C (104°F) in some areas, causing heat-related illnesses and discomfort.
	+ The heat wave was particularly severe in the Kanto region, which includes Tokyo.
	+ Some transportation services were disrupted due to heat-related issues with train and bus systems.
* **Financial Impact:** No financial impact data is available or recorded for this event.

It's worth noting that Japan experiences a hot summer season each year, but the 2019 heat wave was particularly severe due to a prolonged period of high temperatures. The disaster highlighted the need for effective heat management strategies in urban areas, especially during periods of extreme weather

# ── Pipeline 2: Hybrid retrieval (embedding + TF-IDF) + BART summarisation ──

In [23]:
# ── Pipeline 2 — Central Configuration ───────────────────────────────────────
# All tunable parameters are kept here so you never have to hunt through the
# notebook to change a threshold, weight, or word list.

# ---------- Models -----------------------------------------------------------
SUMMARIZER_MODEL     = "facebook/bart-large-cnn"
CROSS_ENCODER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ---------- Retrieval --------------------------------------------------------
HYBRID_WEIGHT_SEMANTIC = 0.65   # bi-encoder semantic similarity weight
HYBRID_WEIGHT_TFIDF    = 0.35   # TF-IDF lexical similarity weight  (must sum to 1)
TOP_K       = 5                 # final events returned to the user
CANDIDATE_K = 20                # candidate pool fed to the cross-encoder

# ---------- Summarisation ----------------------------------------------------
# max summary length  = min(SUMMARY_MAX_TOKENS, input_words * SUMMARY_MAX_RATIO)
# min summary length  = max_length * SUMMARY_MIN_RATIO
SUMMARY_MAX_TOKENS = 150
SUMMARY_MIN_RATIO  = 0.67       # min_length = max_length * this value

# ---------- Keyword expansion ------------------------------------------------
# Maps a query word → list of semantically related terms added to the query.
# Add / remove entries here without touching any function code.
DISASTER_SYNONYMS: dict = {
    "earthquake": ["seismic activity", "quake", "tremor", "seismic"],
    "flood":      ["inundation", "deluge", "high water", "flooding", "flash flood"],
    "hurricane":  ["cyclone", "typhoon", "tropical storm", "storm surge"],
    "tornado":    ["twister", "funnel cloud", "wind storm"],
    "volcano":    ["eruption", "lava flow", "volcanic explosion", "volcanic activity"],
    "volcanic":   ["eruption", "lava flow", "volcanic activity"],
    "wildfire":   ["bushfire", "forest fire", "wildland fire"],
    "tsunami":    ["tidal wave", "seismic sea wave"],
    "drought":    ["water scarcity", "dry spell", "arid conditions"],
    "landslide":  ["mudslide", "rockslide", "debris flow"],
    "cyclone":    ["hurricane", "typhoon", "tropical storm"],
    "storm":      ["hurricane", "cyclone", "typhoon", "tropical storm"],
    "fire":       ["wildfire", "bushfire", "forest fire"],
    "explosion":  ["blast", "detonation", "industrial accident"],
    "epidemic":   ["outbreak", "pandemic", "disease spread"],
}

# ---------- Factual-consistency stopwords ------------------------------------
# Tokens filtered out when computing coverage / hallucination rate.
# Extend freely — adding noisy domain tokens here improves metric reliability.
CONSISTENCY_STOPWORDS: set = {
    # common English function words
    "the", "a", "an", "of", "in", "and", "or", "to", "is", "was",
    "with", "for", "are", "its", "it", "on", "at", "by", "from",
    "this", "that", "be", "has", "have", "been", "were", "as", "no",
    # domain / formatting noise
    "nan", "000", "usd", "us$", "k", "recorded", "data", "impact",
    "financial", "classification", "specific", "location", "occurred",
    "event", "disaster",
}

In [24]:
import ast
import re

import pandas as pd
import spacy
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_huggingface import HuggingFaceEmbeddings
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score

In [25]:
# Load spaCy's Named Entity Recognition model (NER)
nlp = spacy.load("en_core_web_sm")

# Load the summarization model — controlled by SUMMARIZER_MODEL in the config cell
summarizer = pipeline("summarization", model=SUMMARIZER_MODEL)

Device set to use cpu


In [26]:
# Reuse the model constant defined in Cell 0; instantiate a fresh embedder
hf = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

In [27]:
DATA_PATH = "disaster_info_with_embeddings.csv"
df1 = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")

In [28]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df1['combined'].tolist())

In [29]:
# Function to extract disaster-related entities
# The structured combined text uses a predictable pattern; regex is more reliable
# than spaCy NER labels (which mis-tag "the 'Natural'" as an entity).
def extract_disaster_entities(text):
    """Extract disaster type, location, and severity from structured event text."""
    import re

    disaster_type, location, severity = [], [], []

    # Extract classification from "Classification: X > Y > Z." pattern
    cls_match = re.search(r"Classification:\s*[^>]+>\s*[^>]+>\s*([^.]+)\.", text)
    if cls_match:
        disaster_type.append(cls_match.group(1).strip())

    # Extract country / region from "occurred in COUNTRY (REGION)"
    loc_match = re.search(r"occurred in ([^(]+)\(([^)]+)\)", text)
    if loc_match:
        location.append(loc_match.group(1).strip())
        location.append(loc_match.group(2).strip())

    # Extract financial figures as severity proxies
    for m in re.finditer(r"([\d,]+(?:\.\d+)?)\s*K USD", text):
        severity.append(m.group(0))

    # Fallback: use spaCy for plain-text queries (no structured pattern present)
    if not disaster_type and not location:
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "EVENT":
                disaster_type.append(ent.text)
            elif ent.label_ in ["GPE", "LOC"]:
                location.append(ent.text)
            elif ent.label_ in ["CARDINAL", "QUANTITY", "MONEY"]:
                severity.append(ent.text)

    return {
        "disaster_type": list(dict.fromkeys(disaster_type)),
        "location": list(dict.fromkeys(location)),
        "severity": severity,
    }


# Function for keyword expansion — synonym table lives in the config cell above
def keyword_expansion(query, synonyms: dict = None):
    """
    Expands query keywords using disaster-specific synonyms.
    Uses DISASTER_SYNONYMS from config by default; pass a custom dict to override.
    """
    if synonyms is None:
        synonyms = DISASTER_SYNONYMS

    expanded_query = [query]
    for word in query.lower().split():
        # Strip trailing punctuation before lookup
        clean_word = word.strip(".,!?;:")
        if clean_word in synonyms:
            expanded_query.extend(synonyms[clean_word])

    print(f"Expanded Query: {expanded_query}")
    return " ".join(expanded_query)

In [30]:
# ── Cross-encoder for reranking ────────────────────────────────────────────────
from sentence_transformers import CrossEncoder

# Model controlled by CROSS_ENCODER_MODEL in the config cell
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)


# Helper utilities ─────────────────────────────────────────────────────────────
def extract_dates_from_text(event_text):
    """Extracts the first 4-digit year from an event description."""
    match = re.search(r'\b(19|20)\d{2}\b', event_text)
    return int(match.group(0)) if match else None


def extract_year_from_query(query):
    """Extracts a specific year if mentioned in the query."""
    match = re.search(r'\b(19|20)\d{2}\b', query)
    return int(match.group(0)) if match else None


def sort_events_by_date(events):
    """Sort events most-recent-first based on extracted year."""
    dated = [(e, extract_dates_from_text(e)) for e in events]
    dated = [d for d in dated if d[1] is not None]
    dated.sort(key=lambda x: x[1], reverse=True)
    return [d[0] for d in dated]


# ── Main retrieval function ────────────────────────────────────────────────────
def retrieve_relevant_events(
    query,
    embeddings,
    events,
    top_k: int = None,
    candidate_k: int = None,
    semantic_weight: float = None,
    tfidf_weight: float = None,
):
    """
    Three-stage retrieval:
      1. Hybrid bi-encoder (semantic + TF-IDF) → candidate_k candidates
      2. Cross-encoder reranking                → top_k results
      3. Optional year filter

    All defaults are pulled from the config cell; pass explicit values to override
    for a specific call without changing global settings.
    """
    top_k          = top_k          if top_k          is not None else TOP_K
    candidate_k    = candidate_k    if candidate_k    is not None else CANDIDATE_K
    semantic_weight = semantic_weight if semantic_weight is not None else HYBRID_WEIGHT_SEMANTIC
    tfidf_weight   = tfidf_weight   if tfidf_weight   is not None else HYBRID_WEIGHT_TFIDF

    expanded_query = keyword_expansion(query)
    specified_year = extract_year_from_query(query)

    # Stage 1a – semantic similarity
    query_embedding = hf.embed_documents([expanded_query])[0]
    parsed_embeddings = [ast.literal_eval(e) for e in embeddings]
    sem_scores = cosine_similarity([query_embedding], parsed_embeddings).flatten()

    # Stage 1b – TF-IDF lexical similarity
    tfidf_query_vec = tfidf_vectorizer.transform([expanded_query])
    tfidf_scores = cosine_similarity(tfidf_query_vec, tfidf_matrix).flatten()

    # Hybrid combination — weights from config (or caller override)
    hybrid_scores = semantic_weight * sem_scores + tfidf_weight * tfidf_scores

    # Pull top candidate_k for the cross-encoder to rerank
    candidate_indices = hybrid_scores.argsort()[-candidate_k:][::-1]
    candidates = [events[i] for i in candidate_indices]

    # Stage 2 – cross-encoder reranking
    ce_pairs = [[query, c] for c in candidates]
    ce_scores = cross_encoder.predict(ce_pairs)
    reranked = sorted(zip(ce_scores, candidates), key=lambda x: x[0], reverse=True)
    retrieved_events = [evt for _, evt in reranked[:top_k]]

    # Stage 3 – year filter (applied after reranking to avoid over-pruning)
    if specified_year:
        year_filtered = [e for e in retrieved_events if extract_dates_from_text(e) == specified_year]
        if year_filtered:
            retrieved_events = year_filtered

    print(f"\nRetrieved Events (Top {min(top_k, len(retrieved_events))}):")
    for i, event in enumerate(retrieved_events):
        print(f"{i+1}. {event[:110]}...")

    return retrieved_events


def entity_linking(events):
    """Cluster similar disaster events by type + location."""
    linked_events: dict = {}
    for event in events:
        entities = extract_disaster_entities(event)
        key = (tuple(entities["disaster_type"]), tuple(entities["location"]))
        linked_events.setdefault(key, []).append(event)
    return linked_events


# ── Run retrieval ──────────────────────────────────────────────────────────────
# Change this query to test different lookups.
# Examples: "recent volcano in Indonesia", "flood in Bangladesh 2022", "earthquake in Turkey"
query = "recent volcano in Indonesia"

retrieved_events = retrieve_relevant_events(
    query, df1["embeddings"].tolist(), df1["combined"].tolist()
)

Expanded Query: ['recent volcano in Indonesia', 'eruption', 'lava flow', 'volcanic explosion', 'volcanic activity']

Retrieved Events (Top 5):
1. Event 'Lewotolo Volcano' occurred in Indonesia (South-eastern Asia), from 2020-11-29 to 2020-12-31. Classifica...
2. Event 'Ruang volcano' occurred in Indonesia (South-eastern Asia), from 2024-4-16 to 2024-4-29. Classification:...
3. Event 'Ibu volcano' occurred in Indonesia (South-eastern Asia), from 2024-5-16 to 2024-6-6. Classification: Na...
4. Event 'Volcano 'Lewotobi Laki-Laki'' occurred in Indonesia (South-eastern Asia), from 2023-12-Unknown to 2024-...
5. Event 'Mt. Agung' occurred in Indonesia (South-eastern Asia), from 2017-9-22 to 2017-11-28. Classification: Na...


In [31]:
import re as _re

def clean_event_text(text: str) -> str:
    """Remove literal 'nan' tokens and tidy up whitespace for BART input."""
    text = _re.sub(r"\bnan\b", "", text, flags=_re.IGNORECASE)
    text = _re.sub(r"\(\s*\)", "", text)
    text = _re.sub(r":\s*\.", ".", text)
    text = _re.sub(r",\s*;", ";", text)
    text = _re.sub(r"[ \t]{2,}", " ", text)
    text = _re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# ── NER + entity linking (over all retrieved events for context) ──────────────
ner_results = extract_disaster_entities(" ".join(retrieved_events))
print("Extracted Disaster Information:", ner_results)

linked_disasters = entity_linking(retrieved_events)

# ── Choose primary event for summarisation ────────────────────────────────────
# retrieved_events[0] is already the highest cross-encoder scored result —
# the single most relevant answer to the query.
# Summarising ONE event avoids BART picking a random event from a long concat,
# which caused the mismatch between generated summary and reference.
primary_event = retrieved_events[0]
primary_clean  = clean_event_text(primary_event)

print(f"\nPrimary event (top cross-encoder result) for summarisation:")
print(primary_clean[:400])

# Build the full cleaned text for display / evaluation reference
cleaned_events = [clean_event_text(e) for e in retrieved_events]
relevant_text_representation = "\n\n".join(cleaned_events)

# ── BART summarisation ────────────────────────────────────────────────────────
# Lengths are derived from config values: SUMMARY_MAX_TOKENS, SUMMARY_MIN_RATIO
input_word_count = len(primary_clean.split())
safe_max_length  = max(60, min(SUMMARY_MAX_TOKENS, input_word_count // 2))
safe_min_length  = max(10, int(safe_max_length * SUMMARY_MIN_RATIO))

summary = summarizer(
    primary_clean,
    max_length=safe_max_length,
    min_length=safe_min_length,
    do_sample=False,
    truncation=True,
)
summary_text = summary[0]["summary_text"]

bullet_summary = "\n- " + "\n- ".join(
    s.strip() for s in summary_text.split(". ") if s.strip()
)
print("\nFinal Summary of Top Disaster Event:", bullet_summary)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Extracted Disaster Information: {'disaster_type': ['Volcanic activity'], 'location': ['Indonesia', 'South-eastern Asia'], 'severity': []}

Primary event (top cross-encoder result) for summarisation:
Event 'Lewotolo Volcano' occurred in Indonesia (South-eastern Asia), from 2020-11-29 to 2020-12-31. Classification: Natural > Geophysical > Volcanic activity. Specific location: Lembata Regency, Solor Archipelago. No financial impact data recorded.

Final Summary of Top Disaster Event: 
- Event 'Lewotolo Volcano' occurred in Indonesia (South-eastern Asia), from 2020-11-29 to 2020-12-31
- Specific location: Lembata Regency, Solor Archipelago
- No financial impact data recorded.


In [32]:
# ── Evaluation ────────────────────────────────────────────────────────────────
# Reference = the SAME single event that BART was asked to summarise (primary_clean).
# Both generated summary and reference now describe the same disaster, so
# ROUGE/BLEU/BERTScore reflect true faithfulness rather than penalising BART for
# not covering all 5 retrieved events.
reference_summary = primary_clean

print("\nGenerated Summary:\n", summary_text)
print("\nReference (primary event, first 500 chars):\n", reference_summary[:500])

# ROUGE
rouge = Rouge()
rouge_scores = rouge.get_scores(summary_text, reference_summary, avg=True)

# BLEU (sentence-level; low by design because summaries are abstractive)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smoother = SmoothingFunction().method4
bleu_score = sentence_bleu(
    [reference_summary.split()],
    summary_text.split(),
    smoothing_function=smoother,
)

# BERTScore
P, R, F1 = score([summary_text], [reference_summary], lang="en")

print("\n── Automatic Metrics ───────────────────────────────")
print(f"ROUGE-1  F1 : {rouge_scores['rouge-1']['f']:.4f}  "
      f"(P={rouge_scores['rouge-1']['p']:.4f}, R={rouge_scores['rouge-1']['r']:.4f})")
print(f"ROUGE-2  F1 : {rouge_scores['rouge-2']['f']:.4f}  "
      f"(P={rouge_scores['rouge-2']['p']:.4f}, R={rouge_scores['rouge-2']['r']:.4f})")
print(f"ROUGE-L  F1 : {rouge_scores['rouge-l']['f']:.4f}  "
      f"(P={rouge_scores['rouge-l']['p']:.4f}, R={rouge_scores['rouge-l']['r']:.4f})")
print(f"BLEU (smoothed): {bleu_score:.6f}")
print(f"BERTScore — P: {P.mean().item():.4f}, R: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")


def check_factual_consistency(summary: str, source_texts: list, stopwords: set = None) -> dict:
    """
    Token-overlap factual consistency check.
    • Coverage  = fraction of key source tokens that appear in the summary.
    • Hallucination rate = fraction of summary tokens absent from the source.

    Stopwords are filtered before comparison.  Uses CONSISTENCY_STOPWORDS from
    the config cell by default; pass a custom set to override for a specific call.
    """
    stop = stopwords if stopwords is not None else CONSISTENCY_STOPWORDS

    source_blob   = " ".join(source_texts).lower()
    summary_lower = summary.lower()

    def tokenise(text):
        return {t for t in _re.split(r"[\s,;.!?()'\"]", text) if len(t) > 2 and t not in stop}

    source_tokens  = tokenise(source_blob)
    summary_tokens = tokenise(summary_lower)

    missing      = sorted(source_tokens - summary_tokens)
    hallucinated = sorted(summary_tokens - source_tokens)

    coverage           = len(source_tokens & summary_tokens) / max(len(source_tokens), 1)
    hallucination_rate = len(hallucinated) / max(len(summary_tokens), 1)

    return {
        "Coverage (source tokens in summary)": f"{coverage:.2%}",
        "Hallucination rate (novel summary tokens)": f"{hallucination_rate:.2%}",
        "Missing key terms (first 15)": missing[:15],
        "Novel terms in summary (first 15)": hallucinated[:15],
    }

print("\n── Factual Consistency ─────────────────────────────")
# Pass only the primary event so coverage/hallucination are measured against
# the single document that BART summarised.
fc = check_factual_consistency(summary_text, [primary_event])
for k, v in fc.items():
    print(f"{k}: {v}")


Generated Summary:
 Event 'Lewotolo Volcano' occurred in Indonesia (South-eastern Asia), from 2020-11-29 to 2020-12-31. Specific location: Lembata Regency, Solor Archipelago. No financial impact data recorded.

Reference (primary event, first 500 chars):
 Event 'Lewotolo Volcano' occurred in Indonesia (South-eastern Asia), from 2020-11-29 to 2020-12-31. Classification: Natural > Geophysical > Volcanic activity. Specific location: Lembata Regency, Solor Archipelago. No financial impact data recorded.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



── Automatic Metrics ───────────────────────────────
ROUGE-1  F1 : 0.8846  (P=1.0000, R=0.7931)
ROUGE-2  F1 : 0.8235  (P=0.9545, R=0.7241)
ROUGE-L  F1 : 0.8846  (P=1.0000, R=0.7931)
BLEU (smoothed): 0.682749
BERTScore — P: 0.9913, R: 0.9571, F1: 0.9739

── Factual Consistency ─────────────────────────────
Coverage (source tokens in summary): 70.59%
Hallucination rate (novel summary tokens): 0.00%
Missing key terms (first 15): ['activity', 'classification:', 'geophysical', 'natural', 'volcanic']
Novel terms in summary (first 15): []
